## Initial Installs 

In [ ]:
%pip install torch torchvision scikit-learn numpy matplotlib

## Import Libraries

In [12]:
import argparse
import csv
import gc 
import json
import random
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np

import torch
from torch import nn
from torch.utils.data import (
    DataLoader, Subset) 
from torchvision import (
    datasets, models, transforms as T)

from sklearn.metrics import(
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.model_selection import StratifiedKFold

## Seed Creation for Reproducibility

In [ ]:
def experiment_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def synchronize(device):
    device = torch.device(device)
    if device.type == "cude":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()
    print(f"Using {device}")  

## CNN With Incorporated Self Attention Mechanism and MLP

In [10]:
class SelfAttentionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
           # 32 3 x 3 filter to detect simple features 
           nn.Conv2d(1, 32, kernel_size=3, padding=1),
           nn.BatchNorm2d(32),
           nn.ReLU(),
           nn.MaxPool2d(2),

           # channel increate from 32 to 64 to detect more medium level features 
           nn.Conv2d(32, 64, kernel_size=3, padding=1),
           nn.BatchNorm2d(64),
           nn.ReLU(),
           nn.MaxPool2d(2),
           

           # channel increate from 64 to 128 detect more high level features 
           nn.Conv2d(64, 128, kernel_size=3, padding=1),
           nn.BatchNorm2d(128),
           nn.ReLU(),
        )
        
        # positional embedding for the 49 spatial tokens
        self.position =  nn.Parameter(torch.zeros(1, 49, 128))
        nn.init.normal_(self.position, std=0.02)

        # self attention over spatial patches
        self.attention = nn.MultiheadAttention(
            embed_dim= 128,
            num_heads= 4,
            dropout= 0.1, #intial dropout rate will be .1 may increase if generalization needs improvement
            batch_first= True,
        )

        # normalization for 128 dims
        self.norm = nn.LayerNorm(128)

        # classification head that turns 128 features into class logits
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes),
        )

    def forward(self, images):
            # pulls cnn feature map
            features = self.features(images)
            
            # reshape to patch sequence
            x = features.flatten(2).transpose(1,2)
            x = x + self.position

            attended, _ = self.attention(
                x, x, x, need_weights= False
            )

            x = self.norm(x + attended)
            pooled = x.mean(dim= 1)

            return self.classifier(pooled)

## Model Constructor

In [ ]:
def make_model(name):
    if name == "attention_cnn":
        return SelfAttentionCNN(num_classes=10)
    if name == "efficientnet_b3":
        model = models.efficientnet_b3(
        weights = models.EfficientNet_B3_Weights.DEFAULT
    )
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features, 10
    )
    # all layers trainable for entire session
    return model 

## Transformation Definitions

In [11]:
def make_transformation(name):
    if name == "attention_cnn":
        return T.Compose([T.ToTensor(), T.Normalize((0.5), (0.5)),])
    
    if name == "efficientnet_b3":
        return T.Compose([T.Grayscale(num_output_channels=3), models.EfficientNet_B3_Weights.DEFAULT.transforms()])
    raise ValueError(f"Invalid Model: {name}")

## Dataloader

In [ ]:
def data_loader(dataset, config, shuffle= False, seed= 420):
    return DataLoader(dataset, batch_size= config["batch_size"], 
                      shuffle=shuffle, num_workers=config["workers"], 
                      pin_memory=config["device"].type == "cuda",
                      generator=torch.Generator().manual_seed(seed))

## Training

In [ ]:
def train_epoch(model, loader, device, optimizer=None):
    
    is_training = optimizer is not None
    model.train(is_training)

    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    correct = 0
    count = 0

    #synchronizes gpu before starting timer
    synchronize(device)
    start_time = time.perf_counter()

    with torch.set_grad_enabled(is_training):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if is_training:
                optimizer.zero_grad(set_to_none=True)

            # forward pass and loss calculation
            logits = model(images)
            loss = criterion(logits, labels)

            # backpropagation and weight updates
            if is_training:
                loss.backward()
                optimizer.step()

            # gather weighted loss and accuracy
            batch_size = labels.size(0)
            total_loss += loss.detach().item() * batch_size
            correct += (logits.argmax(dim=1) == labels).sum().item()
            count += batch_size
    
    # synchronize gpu before stopping timer
    synchronize(device)
    stop_time = time.perf_counter() - start_time

    return total_loss / count, correct / count, stop_time

## Model Prediction Calculation

In [14]:
#remove autograd tracking to make inference faster
@torch.inference_mode()
def model_prediction(model, data_loader, device):
    
    #set model to eval mode removing things like dropout
    model.eval()

    true_values = []
    predictions = []

    for images, labels, in data_loader:

        # forward pass on gpu device
        logits = model(images.to(device))

        # turn prediction class indices/true lavels into list
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        true_values.extend(labels.tolist())
        
    return np.asarray(true_values), np.asarray(predictions)

## Metric Calculation

In [13]:
def metric_calculation(y_true, y_prediction):
    precision, recall, f1, _ = precision_recall_fscore_support( y_true, y_prediction, labels= list(range(10),
                                                                average = "macro", zero_division = 0))
    return {"accuracy": float(accuracy_score(y_true, y_prediction)),
            "precision_macro": float(precision), "recall_macro": float(recall), "f1_macro": float(f1)}

def print_to_csv(path, rows): 
    with Path(path).open("W", newline="") as file:
        printer = csv.DictWriter(file, fieldnames=list(rows[0]))
        printer.writeheader()
        printer.writerow(rows)
        

## Plotting Functions

In [16]:
def curve_plotting(history, path, include_validation=True):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    epochs = [row["epoch"] for row in history]

    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(epochs, [row[f"train_{metric}"] for row in history], label = "Training")
    
        if include_validation: 
            ax.plot(epochs, [row[f"val_{metric}"] for row in history], label="Validation")
        
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric.title())
        ax.legend()
    
    fig.tight_layout()
    fig.savefig(path, dpi= 160)
    plt.close(fig)

def save_plots(dataset, y_true, y_predicition, classes, directory):
    conf_matrix = confusion_matrix(y_true, y_predicition, labels=list(range(10)))
    fig, axis = plt.subplots(figsize=(10, 9))

    ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=classes).plot(ax=axis, xticks_rotation=45, colorbar=False)

    fig.tight_layout()
    fig.savefig(directory / "confusion_matrix.png", dpi=160)
    plt.close(fig)

    for show_correct, name in [(True, "correct"), (False, "incorrect")]:
        indices = np.flatnonzero((y_true == y_predicition) == show_correct)[:12]

        fig, axes = plt.subplots(3, 4, figsize= (12, 9))

        for axis in axes.flat:
            axis.axis("off")

        for axis, idx in zip(axes.flat, indices):
            axis.imshow(dataset.data[int(idx)].numpy(), cmap="gray")

            axis.set_title(f"True: {classes[int(y_true[idx])]}\n",
                           f"Prediction: {classes[int(y_predicition[idx])]}\n", fontsize= 9)
            
            fig.tight_layout()
            fig.savefig(directory / f"{name}_prediction.png", dpi=160)
            plt.close(fig)
